# Other LLM APIs

**Module:** 08-llm-apis

**Notebook:** `07-other-llm-apis.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **Why Multiple Providers?** with clear contracts and failure modes
- Explain and apply **Groq** with clear contracts and failure modes
- Explain and apply **Together AI** with clear contracts and failure modes
- Explain and apply **OpenRouter** with clear contracts and failure modes
- Explain and apply **Cohere** with clear contracts and failure modes
- Explain and apply **Hugging Face Inference** with clear contracts and failure modes
- Explain and apply **Multi-Provider Router Stub** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — Other LLM APIs

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **Why Multiple Providers?**
2. **Groq**
3. **Together AI**
4. **OpenRouter**
5. **Cohere**
6. **Hugging Face Inference**
7. **Multi-Provider Router Stub**

Read top-to-bottom once, then revisit weak spots with the exercises.


## Why Multiple Providers?

### Definition
**Why Multiple Providers?** is a core building block in 07-other-llm-apis within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Why Multiple Providers? typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Why Multiple Providers?: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Why Multiple Providers? as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Why Multiple Providers? as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Why Multiple Providers?
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Why Multiple Providers? when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does Why Multiple Providers? improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "Why Multiple Providers?" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Why Multiple Providers?"
    notebook: str = "07-other-llm-apis"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
import os

def build_chat_request(model: str, user: str, system: str = "You are concise."):
    return {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "temperature": 0.2,
    }

headers = {"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY')}", "Content-Type": "application/json"}
fake_response = {
    "id": "chatcmpl_demo",
    "choices": [{"message": {"role": "assistant", "content": "OK"}, "finish_reason": "stop"}],
    "usage": {"prompt_tokens": 42, "completion_tokens": 1, "total_tokens": 43},
}
print(build_chat_request("gpt-4.1-mini", "ping")["model"])
print("auth:", headers["Authorization"][:20] + "...", "usage:", fake_response["usage"])


In [ ]:
# Multi-provider router stub
PROVIDERS = {
    "openai": {"base": "https://api.openai.com/v1", "env": "OPENAI_API_KEY"},
    "anthropic": {"base": "https://api.anthropic.com/v1", "env": "ANTHROPIC_API_KEY"},
    "gemini": {"base": "https://generativelanguage.googleapis.com", "env": "GOOGLE_API_KEY"},
}

def resolve_provider(name: str) -> dict:
    p = PROVIDERS[name]
    import os
    return {"base": p["base"], "api_key": os.environ.get(p["env"], "YOUR_API_KEY")}

print({k: resolve_provider(k)["api_key"][:12] + "..." for k in PROVIDERS})


## Groq

### Definition
**Groq** is a core building block in 07-other-llm-apis within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Groq typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Groq: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Groq as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Groq as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Groq
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Groq when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Groq" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Groq"
    notebook: str = "07-other-llm-apis"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
import os

def build_chat_request(model: str, user: str, system: str = "You are concise."):
    return {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "temperature": 0.2,
    }

headers = {"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY')}", "Content-Type": "application/json"}
fake_response = {
    "id": "chatcmpl_demo",
    "choices": [{"message": {"role": "assistant", "content": "OK"}, "finish_reason": "stop"}],
    "usage": {"prompt_tokens": 42, "completion_tokens": 1, "total_tokens": 43},
}
print(build_chat_request("gpt-4.1-mini", "ping")["model"])
print("auth:", headers["Authorization"][:20] + "...", "usage:", fake_response["usage"])


In [ ]:
# Multi-provider router stub
PROVIDERS = {
    "openai": {"base": "https://api.openai.com/v1", "env": "OPENAI_API_KEY"},
    "anthropic": {"base": "https://api.anthropic.com/v1", "env": "ANTHROPIC_API_KEY"},
    "gemini": {"base": "https://generativelanguage.googleapis.com", "env": "GOOGLE_API_KEY"},
}

def resolve_provider(name: str) -> dict:
    p = PROVIDERS[name]
    import os
    return {"base": p["base"], "api_key": os.environ.get(p["env"], "YOUR_API_KEY")}

print({k: resolve_provider(k)["api_key"][:12] + "..." for k in PROVIDERS})


### Worked scenario — Groq

**Situation:** A team wants to productionize a feature involving **Groq**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Together AI

### Definition
**Together AI** is a core building block in 07-other-llm-apis within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Together AI typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Together AI: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Together AI as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Together AI as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Together AI
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Together AI when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Together AI" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Together AI"
    notebook: str = "07-other-llm-apis"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Together AI"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Together AI"}
strong = {"definition": "Together AI", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Together AI"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Together AI", "passed": len(checks)-len(failed), "failed": failed})


## OpenRouter

### Definition
**OpenRouter** is a core building block in 07-other-llm-apis within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around OpenRouter typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For OpenRouter: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain OpenRouter as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating OpenRouter as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for OpenRouter
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use OpenRouter when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "OpenRouter" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "OpenRouter"
    notebook: str = "07-other-llm-apis"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


### Worked scenario — OpenRouter

**Situation:** A team wants to productionize a feature involving **OpenRouter**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Cohere

### Definition
**Cohere** is a core building block in 07-other-llm-apis within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Cohere typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Cohere: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Cohere as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Cohere as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Cohere
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Cohere when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Cohere" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Cohere"
    notebook: str = "07-other-llm-apis"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
import os

def build_chat_request(model: str, user: str, system: str = "You are concise."):
    return {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "temperature": 0.2,
    }

headers = {"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY')}", "Content-Type": "application/json"}
fake_response = {
    "id": "chatcmpl_demo",
    "choices": [{"message": {"role": "assistant", "content": "OK"}, "finish_reason": "stop"}],
    "usage": {"prompt_tokens": 42, "completion_tokens": 1, "total_tokens": 43},
}
print(build_chat_request("gpt-4.1-mini", "ping")["model"])
print("auth:", headers["Authorization"][:20] + "...", "usage:", fake_response["usage"])


In [ ]:
# Multi-provider router stub
PROVIDERS = {
    "openai": {"base": "https://api.openai.com/v1", "env": "OPENAI_API_KEY"},
    "anthropic": {"base": "https://api.anthropic.com/v1", "env": "ANTHROPIC_API_KEY"},
    "gemini": {"base": "https://generativelanguage.googleapis.com", "env": "GOOGLE_API_KEY"},
}

def resolve_provider(name: str) -> dict:
    p = PROVIDERS[name]
    import os
    return {"base": p["base"], "api_key": os.environ.get(p["env"], "YOUR_API_KEY")}

print({k: resolve_provider(k)["api_key"][:12] + "..." for k in PROVIDERS})


## Hugging Face Inference

### Definition
**Hugging Face Inference** is a core building block in 07-other-llm-apis within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Hugging Face Inference typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Hugging Face Inference: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Hugging Face Inference as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Hugging Face Inference as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Hugging Face Inference
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Hugging Face Inference when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Hugging Face Inference" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Hugging Face Inference"
    notebook: str = "07-other-llm-apis"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Hugging Face Inference"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Hugging Face Inference"}
strong = {"definition": "Hugging Face Inference", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Hugging Face Inference"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Hugging Face Inference", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Hugging Face Inference

**Situation:** A team wants to productionize a feature involving **Hugging Face Inference**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Multi-Provider Router Stub

### Definition
**Multi-Provider Router Stub** is a core building block in 07-other-llm-apis within LLM API integration. Treat it as a networked dependency with SLOs, not a local function call: something you can name, version, test, and operate.

### Why it matters
In LLM API integration, weak designs around Multi-Provider Router Stub typically surface as leaked keys, unbounded retries, and silent partial streams. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Multi-Provider Router Stub: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like client wrappers, retries, and usage meters.

### Intuition
Explain Multi-Provider Router Stub as a networked dependency with SLOs, not a local function call. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Multi-Provider Router Stub as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Multi-Provider Router Stub
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM API integration: leaked keys, unbounded retries, and silent partial streams

### When to use
Use Multi-Provider Router Stub when your product path depends on this concern in LLM API integration. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Multi-Provider Router Stub" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Multi-Provider Router Stub"
    notebook: str = "07-other-llm-apis"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_6 = ConceptContract()
print(json.dumps({"contract": asdict(contract_6), "health": contract_6.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


## Comparison Snapshot

Use this table when reviewing designs in **Other LLM APIs**.

| Topic | Do | Don't |
|-------|----|-------|
| Why Multiple Providers? | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Groq | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Together AI | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| OpenRouter | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Cohere | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Hugging Face Inference | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| Why Multiple Providers? | Key concept covered in this notebook; see its section for definition and pitfalls |
| Groq | Key concept covered in this notebook; see its section for definition and pitfalls |
| Together AI | Key concept covered in this notebook; see its section for definition and pitfalls |
| OpenRouter | Key concept covered in this notebook; see its section for definition and pitfalls |
| Cohere | Key concept covered in this notebook; see its section for definition and pitfalls |
| Hugging Face Inference | Key concept covered in this notebook; see its section for definition and pitfalls |
| Multi-Provider Router Stub | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **Other LLM APIs** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **08-llm-apis**.


## Try It Yourself

1. Implement a failing test/fixture for **Why Multiple Providers?**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Groq**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Together AI**, then fix your demo until it passes.
4. Implement a failing test/fixture for **OpenRouter**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Cohere**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
